In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchaudio
import torchvision.models as models
from tqdm import tqdm

# 1. Pipeline Hyperparameters
DATA_DIR = "/kaggle/input/datasets/rheincama/dataset-1-tb-gatekeeper/DATASET_1_TB_GATEKEEPER"
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001
SAMPLE_RATE = 16000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Executing workspace pipeline on device: {device}")

# 2. Custom Spectrogram Audio Dataset
class TBGatekeeperDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.classes = ["Non_TB", "TB"]
        self.file_list = []
        
        for class_idx, class_name in enumerate(self.classes):
            class_folder = os.path.join(root_dir, class_name)
            for fname in os.listdir(class_folder):
                if fname.lower().endswith('.wav'):
                    self.file_list.append((os.path.join(class_folder, fname), class_idx))
                    
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=1024,
            hop_length=256,
            n_mels=64
        ).to(device)
        
        self.db_transform = torchaudio.transforms.AmplitudeToDB().to(device)

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fpath, label = self.file_list[idx]
        waveform, sr = torchaudio.load(fpath)
        waveform = waveform.to(device)
        
        mel_spec = self.mel_transform(waveform)
        log_mel_spec = self.db_transform(mel_spec)
        
        log_mel_spec = (log_mel_spec - log_mel_spec.min()) / (log_mel_spec.max() - log_mel_spec.min() + 1e-6)
        rgb_spectrogram = log_mel_spec.repeat(3, 1, 1)
        
        return rgb_spectrogram, label

# 3. Data Loading & Set Partitioning
full_dataset = TBGatekeeperDataset(DATA_DIR)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_set, val_set = random_split(
    full_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

# 4. Model Modification (ResNet18 Adaption)
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2) 
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 5. Core Training Loop with Early Stopping & Best Checkpoint Tracking
best_val_loss = float('inf')
patience = 3
patience_counter = 0

print("\n🎬 Initiating Training for Model 1 (TB Gatekeeper)...")
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_set)
    epoch_acc = (correct_train / total_train) * 100
    
    # Validation Phase
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_set)
    epoch_val_acc = (correct_val / total_val) * 100
    
    print(f"📊 [Epoch {epoch+1:02d}] Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% || Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

    # Checkpoint and Early Stopping Logic
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "tb_gatekeeper_resnet18.pth")
        print(f"💾 Validation improved. Saved best weights to 'tb_gatekeeper_resnet18.pth'")
        patience_counter = 0  
    else:
        patience_counter += 1
        print(f"⚠️ No improvement. Early stopping counter: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(f"🛑 Training stopped early at Epoch {epoch+1}. Loading best saved weights.")
        break

⚡ Executing workspace pipeline on device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



🎬 Initiating Training for Model 1 (TB Gatekeeper)...


Epoch 1/20: 100%|██████████| 147/147 [00:51<00:00,  2.85it/s]


📊 [Epoch 01] Train Loss: 0.2373 | Train Acc: 92.43% || Val Loss: 0.1797 | Val Acc: 93.69%
💾 Validation improved. Saved best weights to 'tb_gatekeeper_resnet18.pth'


...

📊 [Epoch 03] Train Loss: 0.1058 | Train Acc: 95.73% || Val Loss: 0.1098 | Val Acc: 95.56%
⚠️ No improvement. Early stopping counter: 1/3


Epoch 4/20: 100%|██████████| 147/147 [00:18<00:00,  7.78it/s]


📊 [Epoch 04] Train Loss: 0.0901 | Train Acc: 96.69% || Val Loss: 0.2667 | Val Acc: 87.63%
⚠️ No improvement. Early stopping counter: 2/3


Epoch 5/20: 100%|██████████| 147/147 [00:19<00:00,  7.71it/s]


📊 [Epoch 05] Train Loss: 0.0830 | Train Acc: 96.67% || Val Loss: 0.1142 | Val Acc: 95.90%
⚠️ No improvement. Early stopping counter: 3/3
🛑 Training stopped early at Epoch 5. Loading best saved weights.


In [13]:
import gc
import torch

# Delete Model 1 variables to free up RAM
try:
    del model
    del train_loader
    del val_loader
    del optimizer
    print("Variables deleted.")
except NameError:
    print("Variables already cleared.")

# Force Python garbage collection and empty the VRAM cache
gc.collect()
torch.cuda.empty_cache()

print("🧹 GPU Memory cleared! You can now safely run Model 2 in the next cell.")

Variables deleted.
🧹 GPU Memory cleared! You can now safely run Model 2 in the next cell.


In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchvision.models as models
from tqdm import tqdm

# 1. Pipeline Hyperparameters
DATA_DIR = "/kaggle/input/datasets/rheincama/dataset-2-respiratory/DATASET_2_RESPIRATORY"
BATCH_SIZE = 16  
EPOCHS = 30      
LEARNING_RATE = 0.0005 
SAMPLE_RATE = 16000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Dataset Blueprint (Standard Mel-Spectrogram)
class RespiratoryDataset(Dataset):
    def __init__(self, root_dir, is_training=True):
        self.root_dir = root_dir
        self.classes = ["COPD", "Healthy", "Pneumonia"]
        self.is_training = is_training
        self.file_list = []
        for class_idx, class_name in enumerate(self.classes):
            class_folder = os.path.join(root_dir, class_name)
            if not os.path.exists(class_folder): continue
            for fname in os.listdir(class_folder):
                if fname.lower().endswith('.wav'):
                    self.file_list.append((os.path.join(class_folder, fname), class_idx))
                    
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=1024, hop_length=256, n_mels=64
        ).to(device)
        self.db_transform = torchaudio.transforms.AmplitudeToDB().to(device)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=35)

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        fpath, label = self.file_list[idx]
        waveform, _ = torchaudio.load(fpath)
        log_mel = self.db_transform(self.mel_transform(waveform.to(device)))
        if self.is_training:
            log_mel = self.freq_mask(log_mel)
            log_mel = self.time_mask(log_mel)
        norm_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
        return norm_mel.repeat(3, 1, 1), label

# 3. Secure Partitioning
train_full = RespiratoryDataset(DATA_DIR, is_training=True)
val_full = RespiratoryDataset(DATA_DIR, is_training=False)
g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(train_full), generator=g).tolist()
train_split = int(0.8 * len(train_full))
train_loader = DataLoader(torch.utils.data.Subset(train_full, indices[:train_split]), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(torch.utils.data.Subset(val_full, indices[train_split:]), batch_size=BATCH_SIZE, shuffle=False)

# 4. Model Architecture (Standard)
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 3) 
model = model.to(device)

# Standard Loss (No smoothing or weights)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 5. Training Loop
best_val_loss = float('inf')
patience, patience_counter = 6, 0

print(f"\n🎬 Training Model 2 (Reverting to Peak Performance Baseline)...")
for epoch in range(EPOCHS):
    model.train()
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            val_loss += criterion(model(inputs.to(device)), labels.to(device)).item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "respiratory_classifier_resnet18.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print("🛑 Peak baseline model saved.")
        break

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s] 



🎬 Training Model 2 (Reverting to Peak Performance Baseline)...


Epoch 18: 100%|██████████| 59/59 [00:07<00:00,  7.40it/s]


🛑 Peak baseline model saved.


In [5]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16000
BATCH_SIZE = 16

# 1. Dataset Blueprint Definitions
class M1Dataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.classes = ["Non_TB", "TB"]
        self.file_list = []
        for class_idx, class_name in enumerate(self.classes):
            class_folder = os.path.join(root_dir, class_name)
            if not os.path.exists(class_folder): continue
            for fname in os.listdir(class_folder):
                if fname.lower().endswith('.wav'):
                    self.file_list.append((os.path.join(class_folder, fname), class_idx))
        self.mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=1024, hop_length=256, n_mels=64).to(device)
        self.db_transform = torchaudio.transforms.AmplitudeToDB().to(device)
    def __len__(self): return len(self.file_list)
    def __getitem__(self, idx):
        fpath, label = self.file_list[idx]
        waveform, _ = torchaudio.load(fpath)
        log_mel = self.db_transform(self.mel_transform(waveform.to(device)))
        norm_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
        return norm_mel.repeat(3, 1, 1), label

class M2Dataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.classes = ["COPD", "Healthy", "Pneumonia"]
        self.file_list = []
        for class_idx, class_name in enumerate(self.classes):
            class_folder = os.path.join(root_dir, class_name)
            if not os.path.exists(class_folder): continue
            for fname in os.listdir(class_folder):
                if fname.lower().endswith('.wav'):
                    self.file_list.append((os.path.join(class_folder, fname), class_idx))
        self.mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=1024, hop_length=256, n_mels=64).to(device)
        self.db_transform = torchaudio.transforms.AmplitudeToDB().to(device)
    def __len__(self): return len(self.file_list)
    def __getitem__(self, idx):
        fpath, label = self.file_list[idx]
        waveform, _ = torchaudio.load(fpath)
        log_mel = self.db_transform(self.mel_transform(waveform.to(device)))
        norm_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
        return norm_mel.repeat(3, 1, 1), label

def run_report(model_obj, data_loader, class_names, title):
    model_obj.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in data_loader:
            outputs = model_obj(inputs.to(device))
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    print(f"\n==================================================\n📊 {title}\n==================================================")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0))
    cm = confusion_matrix(all_labels, all_preds)
    print("🧱 Confusion Matrix:")
    print("          " + "".join([f"Pred:{c:<12}" for c in class_names]))
    for i, name in enumerate(class_names):
        print(f"True:{name:<7} " + "".join([f"{val:<16}" for val in cm[i]]))

# 2. Re-instantiate Independent Validation Splits
m1_dir = "/kaggle/input/datasets/rheincama/dataset-1-tb-gatekeeper/DATASET_1_TB_GATEKEEPER"
m2_dir = "/kaggle/input/datasets/rheincama/dataset-2-respiratory/DATASET_2_RESPIRATORY"

m1_ds = M1Dataset(m1_dir)
_, m1_val = torch.utils.data.random_split(m1_ds, [int(0.8*len(m1_ds)), len(m1_ds)-int(0.8*len(m1_ds))], generator=torch.Generator().manual_seed(42))
m1_loader = DataLoader(m1_val, batch_size=BATCH_SIZE, shuffle=False)

m2_ds = M2Dataset(m2_dir)
indices = torch.randperm(len(m2_ds), generator=torch.Generator().manual_seed(42)).tolist()
m2_loader = DataLoader(torch.utils.data.Subset(m2_ds, indices[int(0.8*len(m2_ds)):]), batch_size=BATCH_SIZE, shuffle=False)

# 3. Generate Reports from Saved Weights
if os.path.exists("tb_gatekeeper_resnet18.pth"):
    m1 = models.resnet18()
    m1.fc = nn.Linear(m1.fc.in_features, 2)
    m1.load_state_dict(torch.load("tb_gatekeeper_resnet18.pth", map_location=device))
    run_report(m1.to(device), m1_loader, ["Non_TB", "TB"], "Model 1 (TB Gatekeeper)")

if os.path.exists("respiratory_classifier_resnet18.pth"):
    m2 = models.resnet18()
    m2.fc = nn.Linear(m2.fc.in_features, 3)
    m2.load_state_dict(torch.load("respiratory_classifier_resnet18.pth", map_location=device))
    run_report(m2.to(device), m2_loader, ["COPD", "Healthy", "Pneumonia"], "Model 2 (Respiratory Classifier)")


📊 Model 1 (TB Gatekeeper)
              precision    recall  f1-score   support

      Non_TB     0.9874    0.9432    0.9648       581
          TB     0.9465    0.9882    0.9669       591

    accuracy                         0.9659      1172
   macro avg     0.9670    0.9657    0.9658      1172
weighted avg     0.9668    0.9659    0.9658      1172

🧱 Confusion Matrix:
          Pred:Non_TB      Pred:TB          
True:Non_TB  548             33              
True:TB      7               584             

📊 Model 2 (Respiratory Classifier)
              precision    recall  f1-score   support

        COPD     0.9516    0.8082    0.8741        73
     Healthy     0.9231    0.9730    0.9474        74
   Pneumonia     0.8602    0.9302    0.8939        86

    accuracy                         0.9056       233
   macro avg     0.9116    0.9038    0.9051       233
weighted avg     0.9088    0.9056    0.9047       233

🧱 Confusion Matrix:
          Pred:COPD        Pred:Healthy     Pred:Pne

In [4]:
from IPython.display import FileLink

# List of files to generate links for
model_files = ["tb_gatekeeper_resnet18.pth", "respiratory_classifier_resnet18.pth"]

print("📥 Click the links below to download your model weights:")
for file in model_files:
    try:
        display(FileLink(file))
    except FileNotFoundError:
        print(f"⚠️ {file} not found. Please ensure the training finished successfully.")

📥 Click the links below to download your model weights:


/kaggle/working/tb_gatekeeper_resnet18.pth

/kaggle/working/respiratory_classifier_resnet18.pth